# Quicksort and Partitioning


```{contents}
:local:
:depth: 2
```


```{index} Partition Establishes a Boundary, Not a Fully Sorted Array
```



```{index} Work the Two-Scan Partition Before Recursing
```

## Work the Two-Scan Partition Before Recursing


The conceptual smaller/equal/larger picture describes a goal, but the retained implementation uses two inward scans. It does not return a single final pivot position. Use the exact scan rule to determine the next recursive intervals; otherwise a correct-looking diagram can lead to incorrect code.

Start with 5,2,5,1,8 over inclusive indexes zero through four. The middle **position** is two, so the saved pivot value is five. The left scan stops at a value at least five; the right scan stops at a value at most five. If the scans have not crossed, swap those values and advance both scans, even when the values are equal.

| Step | left | right | Action and array |
| --- | ---: | ---: | --- |
| Initial scans | 0 | 3 | Right skips 8; stop at 5 and 1 |
| Swap, then advance | 1 | 2 | 1,2,5,5,8 |
| Next scans | 2 | 2 | Left skips 2; both stop at the middle 5 |
| Self-swap, then advance | 3 | 1 | Array unchanged; scans cross |
| Recur | [0,1] | [3,4] | These are inclusive subranges, not pivot positions |

Partitioning has established separation, not a completely sorted recursive region in general. Here the displayed values happen already to be sorted after the partition. That coincidence does not eliminate the recursive work required by the general algorithm.


In [1]:
using System;
static void QuickSort(int[] data, int low, int high)
{
    if (low >= high) return;
    int left = low, right = high;
    int pivot = data[low + (high - low) / 2];
    while (left <= right)
    {
        while (data[left] < pivot) left++;
        while (data[right] > pivot) right--;
        if (left <= right)
        {
            (data[left], data[right]) = (data[right], data[left]);
            left++; right--;
        }
    }
    if (low < right) QuickSort(data, low, right);
    if (left < high) QuickSort(data, left, high);
}
int[] values = { 5, 2, 5, 1, 8 };
QuickSort(values, 0, values.Length - 1);
Console.WriteLine(string.Join(",", values));
int[] equal = { 4, 4, 4 };
QuickSort(equal, 0, equal.Length - 1);
Console.WriteLine(string.Join(",", equal));


1,2,5,5,8
4,4,4


The fixed pivot value is copied before any swaps; it is not reread through a position that may now contain another value. Advancing both indexes after a swap guarantees progress when both stopped on equal values. The small method assumes a valid requested subrange; empty and singleton ranges return immediately. The retained comparison example includes Shell sort as well and remains available below.

Guided task: trace an all-equal three-element range. Fill left/right before and after each swap, then write the recursive intervals. As a separate variation, put sentinel values outside an interior requested range and predict which elements must remain unchanged.

```{dropdown} Check progress and the subrange contract
For three equal values, swap positions zero and two, then the middle with itself; left becomes two and right zero. Neither recursive interval has more than one element. For subrange sorting, the values outside the requested low/high endpoints are not part of any swap or recursive range and must remain unchanged. Verify that independently, rather than checking only global sortedness.
```

For independent work, preserve every original value and its multiplicity, not merely nondecreasing output. Equal-key records may exchange places, so this QuickSort is not a stable reporting sort. Pivot/input assumptions determine expected performance; the middle index does not guarantee a median value or balanced subproblems.


**Visualize it.** OpenDSA's [Quicksort](https://opendsa-server.cs.vt.edu/OpenDSA/Books/CS3/html/Quicksort.html) module animates the [partition](https://opendsa-server.cs.vt.edu/OpenDSA/Books/CS3/html/Quicksort.html#partition) scans and swaps on one array, then the recursive calls on each side of the pivot.

```{index} Partition Establishes a Boundary, Not a Fully Sorted Array
```

## Partition Establishes a Boundary, Not a Fully Sorted Array



Quicksort separates data according to a pivot policy and recursively orders the resulting regions. The partition contract must say where smaller, equal and larger values may appear and which indexes are returned. Different valid partition algorithms return different kinds of boundaries. Applying one algorithm's recursive endpoints to another partition result can skip items or recurse forever.

For values 6, 2, 5, 1, 4 with pivot value four, a conceptual partition separates values below four (2 and 1), equal to four, and above four (6 and 5). Their internal order need not yet be sorted. Recursion handles the smaller regions. This conceptual three-way picture explains the goal; the preserved implementation's actual partition form determines its exact index updates.

| Stage | What is established | What remains to do |
| --- | --- | --- |
| Before partition | Pivot/comparison policy is known | Classify values |
| After partition | Values lie in allowed regions | Order each nontrivial recursive region |
| After recursive returns | Regions are ordered | Combine their boundary facts |

A progress argument must cover equal values as well as unequal ones. If a partition never advances on equality, an all-equal input can stall. If a recursive call receives the original full interval again, finding a plausible pivot did not guarantee termination. Trace empty, singleton, duplicate-heavy, already ordered and reverse-ordered cases.

Choosing pivots changes work distributions. Repeated highly unbalanced splits can yield quadratic work and linear recursive depth. A claim of expected good behavior needs assumptions about pivot selection or input distribution. It is not justified by one random-looking demonstration. An in-place partition can also reorder equal-key records, so ordinary quicksort is not automatically a stable report sort.

Guided exercise: take the exact partition rule in the example and annotate its scanned/unknown regions after each update. Record the returned boundary and write the two next recursive intervals before executing them. Check that together they cover all values requiring further work and that each nontrivial recursive interval is smaller. Then compare output occurrence counts with the original array; correct ordering alone does not prove nothing was lost.

For related explanation, see the further reading below.[^development-reading]


```{index} Quicksort
```

## Quicksort




```{index} Quicksort
```

Quicksort, also called partition sort, chooses a pivot and rearranges one subrange around it. The left scan skips values smaller than the pivot; the right scan skips larger values. When the scans stop before crossing, exchange the two values and advance both positions. Advancing after a swap is essential when values equal the pivot.

The partition loop maintains these statements: everything before the left scan is at most the pivot, and everything after the right scan is at least the pivot. The unclassified interval shrinks. After the scans cross, recursively sort the remaining left and right subranges. They are not individually sorted by partitioning, but no value in the left group belongs after a value in the right group.

The same method solves smaller ranges. Empty and one-element ranges are base cases. Quicksort averages `O(n log n)` comparisons under suitable pivot/input assumptions, but repeated unbalanced partitions can require `O(n^2)` work and deep recursion. Choosing the middle **position** does not guarantee choosing the median **value**. This implementation is not stable and uses recursive stack space.

```{index} Partition Trace
```

### Partition Trace




```{index} Partition Trace
```

Trace both scan indices, pivot, swaps, and recursive ranges on `{5, 2, 5, 1, 8}`. Repeat with all values equal and explain why the pointers still advance. Test subrange sorting with sentinel values outside the requested range and verify that the sentinels are unchanged. Explain why the pivot index uses `low + (high - low) / 2` rather than adding two potentially large indices.

In [11]:
using System;
using System.Collections.Generic;
static void ShellSort(int[] data, int[] descendingGaps) {
    foreach (int gap in descendingGaps) {
        if (gap <= 0) throw new ArgumentException("Gaps must be positive.");
        for (int start = 0; start < gap && start < data.Length; start++)
            for (int j = start + gap; j < data.Length; j += gap)
                for (int i = j; i >= gap && data[i] < data[i - gap]; i -= gap)
                    (data[i], data[i - gap]) = (data[i - gap], data[i]);
    }
}
static int[] Gaps(int n) {
    List<int> gaps = new List<int>();
    for (int h = 1; h < n;) {
        gaps.Add(h);
        if (h > (n - 1) / 3) break;
        h = 3 * h + 1;
    }
    gaps.Reverse(); return gaps.ToArray();
}
static void QuickSort(int[] data, int low, int high) {
    if (low >= high) return;
    int left = low, right = high, pivot = data[low + (high - low) / 2];
    while (left <= right) {
        while (data[left] < pivot) left++;
        while (data[right] > pivot) right--;
        if (left <= right) {
            (data[left], data[right]) = (data[right], data[left]);
            left++; right--;
        }
    }
    if (low < right) QuickSort(data, low, right);
    if (left < high) QuickSort(data, left, high);
}
int[] original = {9, 8, 7, 6, 5, 4, 3, 2, 1};
int[] shell = (int[])original.Clone(), quick = (int[])original.Clone();
ShellSort(shell, Gaps(shell.Length));
QuickSort(quick, 0, quick.Length - 1);
Console.WriteLine(string.Join(" ", shell));
Console.WriteLine(string.Join(" ", quick));

1 2 3 4 5 6 7 8 9
1 2 3 4 5 6 7 8 9


Both lines print `1 2 3 4 5 6 7 8 9`: the shell sort and the quicksort agree on the sorted result for the test array. The tracing tasks in the paragraph above are for you to do by hand; the cell confirms only that the two implementations produce the same order.

**Common mistake.** Not advancing the scan indexes after a swap when both values equal the pivot. The next iteration swaps the same two elements back, and the loop never terminates; the `left++` and `right--` after the exchange are what move the scans past equal values.


```{rubric} Footnotes
```


[^development-reading]: Further reading: [OpenDSA, Search in Sorted Arrays](https://opendsa-server.cs.vt.edu/OpenDSA/Books/Catalog/html/SortedSearch.html). The worked business scenarios and guided reasoning here are original teaching examples.
